In [8]:
import os
import json
import time
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

### Config

In [9]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE   = "../data/splits/shared_split.json"

MODEL_DIR = "../models/ml_models"
VECTORIZER_FILE = "../models/vectorizer.joblib"
ML_METADATA_FILE = "../models/ml_all_models_metadata.json"
ML_PRED_FILE = "../data/processed/ml_all_test_predictions.csv"
ML_SUMMARY_FILE = "../data/processed/ml_summary.csv"

os.makedirs("../models", exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

total_start_time = time.time()

### Load data

In [10]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

train_df = df[df["row_id"].isin(split_dict["train_ids"])].copy()
val_df   = df[df["row_id"].isin(split_dict["val_ids"])].copy()
test_df  = df[df["row_id"].isin(split_dict["test_ids"])].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nTotal emotion classes:", df["label"].nunique())
print("Emotion labels:", sorted(df["label"].unique().tolist()))

print("\nAll data label counts:")
print(df["label"].value_counts().sort_index())

print("\nTrain label counts:")
print(train_df["label"].value_counts().sort_index())

print("\nVal label counts:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)

Total emotion classes: 6
Emotion labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']

All data label counts:
label
angry      326
disgust    427
fear       395
happy      234
neutral    500
sad        278
Name: count, dtype: int64

Train label counts:
label
angry      228
disgust    299
fear       276
happy      164
neutral    350
sad        195
Name: count, dtype: int64

Val label counts:
label
angry      49
disgust    64
fear       59
happy      35
neutral    75
sad        42
Name: count, dtype: int64

Test label counts:
label
angry      49
disgust    64
fear       60
happy      35
neutral    75
sad        41
Name: count, dtype: int64


### Vectorizer

In [11]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val   = vectorizer.transform(val_df["text"])
X_test  = vectorizer.transform(test_df["text"])

y_train = train_df["label"]
y_val   = val_df["label"]
y_test  = test_df["label"]

### Models

In [12]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "Support Vector Machine": SVC(
        kernel="linear",
        probability=True,
        class_weight="balanced",
        random_state=42
    )
}

summary_rows = []
pred_df = test_df[["row_id", "text", "label"]].copy()
metadata = {}

### Train + Evaluate

In [13]:
for model_name, model in models.items():
    print("\n" + "=" * 50)
    print("Training:", model_name)
    print("=" * 50)

    train_start = time.time()
    model.fit(X_train, y_train)
    train_elapsed = time.time() - train_start

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    val_acc = accuracy_score(y_val, val_pred)
    val_f1_macro = f1_score(y_val, val_pred, average="macro")

    test_acc = accuracy_score(y_test, test_pred)
    test_f1_macro = f1_score(y_test, test_pred, average="macro")
    test_f1_weighted = f1_score(y_test, test_pred, average="weighted")

    print(f"\nTraining time: {train_elapsed:.2f} seconds")

    print("\nValidation")
    print("Accuracy    :", val_acc)
    print("F1 Macro    :", val_f1_macro)

    print("\nTest")
    print("Accuracy    :", test_acc)
    print("F1 Macro    :", test_f1_macro)
    print("F1 Weighted :", test_f1_weighted)
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, test_pred))

    file_name = model_name.lower().replace(" ", "_") + ".joblib"
    model_path = os.path.join(MODEL_DIR, file_name)
    joblib.dump(model, model_path)

    pred_col = model_name.lower().replace(" ", "_").replace("-", "_") + "_pred"
    pred_df[pred_col] = test_pred

    summary_rows.append({
        "Model": model_name,
        "Accuracy": test_acc,
        "F1_Macro": test_f1_macro,
        "F1_Weighted": test_f1_weighted,
        "Train_Time_Seconds": round(train_elapsed, 4),
        "Has_Predict_Proba": hasattr(model, "predict_proba")
    })

    metadata[model_name] = {
        "model_path": model_path,
        "prediction_column": pred_col,
        "has_predict_proba": hasattr(model, "predict_proba")
    }



Training: Naive Bayes

Training time: 0.01 seconds

Validation
Accuracy    : 0.9074074074074074
F1 Macro    : 0.9207266732717422

Test
Accuracy    : 0.8827160493827161
F1 Macro    : 0.8981922477981571
F1 Weighted : 0.8808117346422094

Classification Report:
              precision    recall  f1-score   support

       angry       0.96      1.00      0.98        49
     disgust       0.85      0.94      0.89        64
        fear       0.83      0.72      0.77        60
       happy       0.97      0.97      0.97        35
     neutral       0.81      0.80      0.81        75
         sad       0.98      0.98      0.98        41

    accuracy                           0.88       324
   macro avg       0.90      0.90      0.90       324
weighted avg       0.88      0.88      0.88       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 1 60  0  0  3  0]
 [ 0  6 43  0 10  1]
 [ 0  0  1 34  0  0]
 [ 1  5  8  1 60  0]
 [ 0  0  0  0  1 40]]

Training: Logistic Regression

Training time: 0.25 s

### Save vectorizer + metadata

In [14]:
joblib.dump(vectorizer, VECTORIZER_FILE)

metadata["labels"] = sorted(df["label"].unique().tolist())
metadata["vectorizer_file"] = VECTORIZER_FILE

with open(ML_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

pred_df.to_csv(ML_PRED_FILE, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["Accuracy", "F1_Macro"],
    ascending=False
).reset_index(drop=True)

summary_df.to_csv(ML_SUMMARY_FILE, index=False, encoding="utf-8-sig")

print("\n===== Summary =====")
print(summary_df)

total_elapsed = time.time() - total_start_time

print("\nSaved:")
print("-", VECTORIZER_FILE)
print("-", ML_METADATA_FILE)
print("-", ML_PRED_FILE)
print("-", ML_SUMMARY_FILE)
print(f"\nTotal training pipeline time: {total_elapsed:.2f} seconds")
print("Done")


===== Summary =====
                    Model  Accuracy  F1_Macro  F1_Weighted  \
0  Support Vector Machine  0.904321  0.919068     0.903556   
1     Logistic Regression  0.895062  0.912052     0.894449   
2             Naive Bayes  0.882716  0.898192     0.880812   
3           Random Forest  0.861111  0.885660     0.863335   

   Train_Time_Seconds  Has_Predict_Proba  
0              3.8756               True  
1              0.2479               True  
2              0.0081               True  
3              8.4738               True  

Saved:
- ../models/vectorizer.joblib
- ../models/ml_all_models_metadata.json
- ../data/processed/ml_all_test_predictions.csv
- ../data/processed/ml_summary.csv

Total training pipeline time: 13.53 seconds
Done
